In [3]:

"""
CURSO: PROYECTO INTEGRADOR - SISTEMAS COMPUTACIONALES (2026-2)
PROCESO: SCRIPT ETL AUTOMATIZADO DATA WAREHOUSE (MINSA - INEN)
AUTORES: Greys Santos, Kiara Cuevas, Kevin Aldana
"""

import pandas as pd
import numpy as np
import logging
import sys


In [ ]:

# 1. CONFIGURACIÓN DEL FRAMEWORK DE BITÁCORA (LOGGING LOCAL)
logging.basicConfig(
    filename='etl_dw_pacientes.log',
    filemode='a',
    format='%(asctime)s - %(levelname)s - [PASO: %(filename)s] - %(message)s',
    level=logging.INFO,
    encoding='utf-8'
)

# Constantes MINSA
SEXO_M = ['M', 'MASCULINO']
SEXO_F = ['F', 'FEMENINO']
BINS_EDAD = [-1, 11, 17, 29, 59, 120]
ETIQUETAS_EDAD = ['0-11', '12-17', '18-29', '30-59', '60+']

def normalizar_ubigeo(serie):
    """Convierte enteros/textos a código estructurado de 6 dígitos."""
    raw = serie.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    largo_ok = raw.str.len() <= 6
    cod = raw.str.zfill(6)
    valido = (cod.str.fullmatch(r"\d{6}") & largo_ok).fillna(False).astype(bool)
    repuestos = int(((raw.str.len() < 6) & valido).fillna(False).sum())
    return cod.where(valido, "000000").astype(object), repuestos, int((~valido).sum())

def normalizar_sexo(serie):
    t = serie.astype("string").str.strip().str.upper()
    es_m = t.isin(SEXO_M).fillna(False).astype(bool)
    es_f = t.isin(SEXO_F).fillna(False).astype(bool)
    out = pd.Series("NO ESPECIFICADO", index=serie.index, dtype=object)
    out[es_m] = "MASCULINO"
    out[es_f] = "FEMENINO"
    return out, int((~(es_m | es_f)).sum())

def calcular_rango_edad(serie):
    e = pd.to_numeric(serie, errors="coerce")
    e = e.where((e >= 0) & (e <= 120))
    r = pd.cut(e, bins=BINS_EDAD, labels=ETIQUETAS_EDAD).astype(object)
    sin_dato = int(e.isna().sum())
    return r.where(r.notna(), "SIN DATO"), sin_dato

def ejecutar_etl():
    id_carga = 1
    logging.info(f"=== INICIANDO SUB-PROCESO ETL CARGA NUMERO: {id_carga} ===")
    print("Iniciando tubería ETL Analítica INEN...")

    try:
        # ==========================================
        # FASE 1: EXTRACT (Extracción Operacional)
        # ==========================================
        logging.info("[EXTRACT] Extrayendo datos desde origen transaccional PACIENTES_IyE...")
        
        # Simulación controlada del volumen de producción del INEN (78,975 Nuevos, 54,636 Egresos)
        vol_nuevos = 78975
        vol_egresos = 54636
        
        logging.info(f"[EXTRACT] Leyendo tabla dbo.INGRESOS. Grano esperado: {vol_nuevos} registros.")
        df_fuente_nuevos = pd.DataFrame({
            'uuid': [f'UUID-N-{i}' for i in range(vol_nuevos)],
            'historia': [f'H-{100000+i}' for i in range(vol_nuevos)],
            'fec_filiacion': np.random.choice(pd.date_range('2025-01-01', '2025-12-31'), vol_nuevos),
            'sexo': np.random.choice(['MASCULINO', 'FEMENINO', 'M', 'F', None], vol_nuevos, p=[0.45, 0.45, 0.04, 0.04, 0.02]),
            'edad': np.random.randint(0, 85, vol_nuevos),
            'ubigeo_lugar_residencia': np.random.choice(['150101', '150132', '70101', '150108'], vol_nuevos, p=[0.4, 0.3, 0.2, 0.1]),
            'lugar_residencia': ['LIMA / LIMA / LIMA'] * vol_nuevos,
            'cant_atenciones_cex': np.random.randint(1, 5, vol_nuevos)
        })

        logging.info(f"[EXTRACT] Leyendo tabla dbo.EGRESOS. Grano esperado: {vol_egresos} registros.")
        df_fuente_egresos = pd.DataFrame({
            'uuid': [f'UUID-E-{i}' for i in range(vol_egresos)],
            'historia': [f'H-{200000+i}' for i in range(vol_egresos)],
            'sexo': np.random.choice(['MASCULINO', 'FEMENINO'], vol_egresos),
            'edad': np.random.randint(0, 85, vol_egresos),
            'fecha_ingreso': pd.date_range('2025-01-01', periods=vol_egresos, freq='min'),
            'fecha_egreso': pd.date_range('2025-01-04', periods=vol_egresos, freq='min'),
            'ubigeo_lugar_residencia': np.random.choice(['150101', '150132', '50101'], vol_egresos),
            'lugar_residencia': ['LIMA / LIMA / LIMA'] * vol_egresos
        })
        
        logging.info("[EXTRACT] Extracción completada sin errores de conexión.")

        # ==========================================
        # FASE 2: TRANSFORM (Limpieza y Enriquecimiento)
        # ==========================================
        logging.info("[TRANSFORM] Iniciando motor de reglas de calidad y anonimización...")

        # --- Transformación Dataset 1 (Ingresos) ---
        logging.info("[TRANSFORM] Procesando subset: INGRESOS.")
        
        filas_antes = len(df_fuente_nuevos)
        df_nuevos = df_fuente_nuevos.drop_duplicates(subset=['uuid', 'historia']).copy()
        duplicados_nuevos = filas_antes - len(df_nuevos)
        if duplicados_nuevos > 0:
            logging.warning(f"[TRANSFORM] Se depuraron {duplicados_nuevos} llaves duplicadas en Ingresos.")

        df_nuevos['cod_ubigeo'], rep_u, rej_u = normalizar_ubigeo(df_nuevos['ubigeo_lugar_residencia'])
        logging.info(f"[TRANSFORM] Ubigeos Ingresos: {rep_u} rellenados con cero, {rej_u} nulos asignados a '000000'.")

        df_nuevos['sexo_norm'], err_sexo = normalizar_sexo(df_nuevos['sexo'])
        df_nuevos['rango_edad'], err_edad = calcular_rango_edad(df_nuevos['edad'])

        df_nuevos['FechaID'] = pd.to_datetime(df_nuevos['fec_filiacion']).dt.strftime('%Y%m%d').astype(int)
        df_nuevos['AtencionID'] = 1  
        df_nuevos['dias_estancia'] = None  
        df_nuevos['cantidad'] = df_nuevos['cant_atenciones_cex']

        # --- Transformación Dataset 2 (Egresos) ---
        logging.info("[TRANSFORM] Procesando subset: EGRESOS.")
        
        filas_antes_e = len(df_fuente_egresos)
        df_egresos = df_fuente_egresos.drop_duplicates(subset=['uuid', 'historia', 'fecha_ingreso']).copy()
        duplicados_egresos = filas_antes_e - len(df_egresos)

        df_egresos['fecha_ingreso'] = pd.to_datetime(df_egresos['fecha_ingreso'])
        df_egresos['fecha_egreso'] = pd.to_datetime(df_egresos['fecha_egreso'])
        validas = df_egresos['fecha_egreso'] >= df_egresos['fecha_ingreso']
        rechazados_fecha = int((~validas).sum())
        df_egresos = df_egresos[validas].copy()
        if rechazados_fecha > 0:
            logging.warning(f"[TRANSFORM] Se rechazaron {rechazados_fecha} filas por inconsistencia operativa.")

        df_egresos['dias_estancia'] = (df_egresos['fecha_egreso'] - df_egresos['fecha_ingreso']).dt.days
        df_egresos['cod_ubigeo'], _, _ = normalizar_ubigeo(df_egresos['ubigeo_lugar_residencia'])
        df_egresos['sexo_norm'], _ = normalizar_sexo(df_egresos['sexo'])
        df_egresos['rango_edad'], _ = calcular_rango_edad(df_egresos['edad'])
        
        df_egresos['FechaID'] = df_egresos['fecha_egreso'].dt.strftime('%Y%m%d').astype(int)
        df_egresos['AtencionID'] = 2  
        df_egresos['cantidad'] = 1  

        # ==========================================
        # FASE 3: LOAD & VALIDATE (Conciliación)
        # ==========================================
        logging.info("[LOAD] Preparando estructura final unificada para Tabla de Hechos...")
        
        columnas_dw = ['FechaID', 'cod_ubigeo', 'rango_edad', 'sexo_norm', 'AtencionID', 'cantidad', 'dias_estancia']
        
        hechos_ingresos = df_nuevos[columnas_dw]
        hechos_egresos = df_egresos[columnas_dw]
        
        total_ingresos_cargados = len(hechos_ingresos)
        total_egresos_cargados = len(hechos_egresos)
        total_combinado = total_ingresos_cargados + total_egresos_cargados

        logging.info(f"[VALIDATE] === CONCILIACIÓN DE INTEGRIDAD FIN DE FLUJO ===")
        logging.info(f"[VALIDATE] PROCESO INGRESOS: Inicial={vol_nuevos} | Duplicados={duplicados_nuevos} | Cargados={total_ingresos_cargados}")
        logging.info(f"[VALIDATE] PROCESO EGRESOS: Inicial={vol_egresos} | Duplicados={duplicados_egresos} | Rechazados Temporal={rechazados_fecha} | Cargados={total_egresos_cargados}")
        logging.info(f"[LOAD] Inserción por bloques simulada exitosamente. Total hechos: {total_combinado} registros.")
        logging.info("=== PROCESO ETL FINALIZADO CON ESTADO [OK] ===")
        
        print("\n¡ETL Completado con éxito!")
        print(f" -> Registros procesados e indexados: {total_combinado}")
        print(" -> Verifique el desglose de auditoría en 'etl_dw_pacientes.log'.")

    except Exception as error:
        logging.error(f"FALTA CRÍTICA - DETENCIÓN INMEDIATA DEL ENGINE ETL: {str(error)}")
        print(f"Error crítico detectado en el pipeline: {str(error)}", file=sys.stderr)

if __name__ == "__main__":
    ejecutar_etl()


Iniciando tubería ETL Analítica INEN...

¡ETL Completado con éxito!
 -> Registros procesados e indexados: 133611
 -> Verifique el desglose de auditoría en 'etl_dw_pacientes.log'.
